# Deep Generative Models - CA1
## Variational Autoencoders (VAE)
### University of Tehran - Electrical and Computer Engineering

**Course:** Deep Generative Models  
**Instructor:** Dr. Mostafa Tavasoli Pour

---

This notebook contains the implementation for CA1 homework covering:
1. **Question 1:** Probabilistic Graphical Models (Bayesian and Markov Networks)
2. **Question 2:** Variational Autoencoders implementation on dSprites dataset

# Question 1: Probabilistic Graphical Models

## Part 1: Bayesian Network - Disease Model

### Sub-part 1: Draw Bayesian Network

Based on the problem description, we have the following variables and their relationships:
- **M**: Immune system strength
- **S**: Season
- **I**: Disease severity  
- **F**: Financial capability
- **T**: Treatment type (expensive vs cheap)
- **D**: Probability of death

**Relationships:**
- Season (S) and Immune system (M) affect Disease severity (I)
- Disease severity (I) affects Probability of death (D)
- Disease severity (I) and Financial capability (F) affect Treatment (T)
- Treatment (T) affects Probability of death (D)

In [ ]:
# Question 1 - Part 1, Sub-part 1: Bayesian Network Structure
# Draw the Bayesian Network using graphviz or networkx

import networkx as nx
from matplotlib.patches import FancyBboxPatch

def draw_bayesian_network():
    """Draw the Bayesian Network for the disease model"""
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Create directed graph
    G = nx.DiGraph()
    
    # Add nodes
    nodes = ['M', 'S', 'I', 'F', 'T', 'D']
    node_labels = {
        'M': 'Immune System\n(M)',
        'S': 'Season\n(S)',
        'I': 'Disease Severity\n(I)',
        'F': 'Financial Capability\n(F)',
        'T': 'Treatment Type\n(T)',
        'D': 'Death Probability\n(D)'
    }
    
    # Add edges based on the problem description
    edges = [
        ('M', 'I'),  # Immune system affects disease severity
        ('S', 'I'),  # Season affects disease severity
        ('I', 'D'),  # Disease severity affects death probability
        ('I', 'T'),  # Disease severity affects treatment choice
        ('F', 'T'),  # Financial capability affects treatment choice
        ('T', 'D'),  # Treatment affects death probability
    ]
    
    G.add_edges_from(edges)
    
    # Position nodes for better visualization
    pos = {
        'M': (0, 2),
        'S': (2, 2),
        'I': (1, 1),
        'F': (3, 1),
        'T': (2, 0),
        'D': (1, -1)
    }
    
    # Draw the graph
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', 
                          node_size=3000, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, node_labels, font_size=9, 
                           font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', 
                          arrows=True, arrowsize=20, 
                          arrowstyle='->', width=2, ax=ax)
    
    ax.set_title('Bayesian Network: Disease Model', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('bayesian_network.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return G, edges

# Draw the network
G, edges = draw_bayesian_network()
print("Bayesian Network Structure:")
print(f"Nodes: {list(G.nodes())}")
print(f"Edges: {edges}")

### Sub-part 2: Joint Probability Distribution

The joint probability distribution for the Bayesian Network can be written as:

$$P(M, S, I, F, T, D) = P(M) \cdot P(S) \cdot P(F) \cdot P(I|M, S) \cdot P(T|I, F) \cdot P(D|I, T)$$

This factorization follows from the conditional independence structure of the Bayesian network.

### Sub-part 3: Conditional Independence Statements

Let's analyze each statement:

In [ ]:
# Question 1 - Part 1, Sub-part 3: Conditional Independence Analysis

def analyze_conditional_independence():
    """
    Analyze conditional independence statements for the Bayesian Network
    """
    statements = {
        'a': {
            'statement': 'F ⊥ D',
            'description': 'F is independent of D (unconditionally)',
            'answer': False,
            'reasoning': 'F and D are NOT independent because there is an active path F → T → D. '
                        'Financial capability affects treatment choice, which affects death probability.'
        },
        'b': {
            'statement': 'S ⊥ D | I',
            'description': 'S is independent of D given I',
            'answer': True,
            'reasoning': 'Given I (disease severity), S (season) is independent of D (death probability). '
                        'Once we know disease severity, knowing the season provides no additional information '
                        'about death probability. The path S → I → D is blocked by observing I.'
        },
        'c': {
            'statement': 'M ⊥ F',
            'description': 'M is independent of F (unconditionally)',
            'answer': True,
            'reasoning': 'M (immune system) and F (financial capability) are independent. '
                        'There is no path connecting them in the graph, and they have no common ancestors.'
        },
        'd': {
            'statement': 'M ⊥ F | T',
            'description': 'M is independent of F given T',
            'answer': False,
            'reasoning': 'Given T (treatment type), M and F become DEPENDENT. This is a V-structure (collider): '
                        'M → I ← S and I → T ← F. Observing T (a descendant of the collider I) '
                        'opens up the path between M and F, creating a dependency.'
        },
        'e': {
            'statement': 'M ⊥ T | {D, I}',
            'description': 'M is independent of T given both D and I',
            'answer': True,
            'reasoning': 'Given both I and D, M is independent of T. '
                        'The path M → I → T is blocked by observing I. '
                        'All information from M about T flows through I, so conditioning on I blocks this path.'
        }
    }
    
    print("="*80)
    print("CONDITIONAL INDEPENDENCE ANALYSIS")
    print("="*80)
    
    for key, data in statements.items():
        print(f"\n{key}. {data['statement']}")
        print(f"   Description: {data['description']}")
        print(f"   Answer: {'TRUE' if data['answer'] else 'FALSE'}")
        print(f"   Reasoning: {data['reasoning']}")
    
    return statements

# Run the analysis
independence_analysis = analyze_conditional_independence()

## Part 2: Given Bayesian Network Analysis

Graph structure:
```
    C
    |
    O - A
    |   |
    S - T - B
        |
        M
```

In [ ]:
# Question 1 - Part 2: Bayesian Network with given structure

def draw_given_bayesian_network():
    """Draw the given Bayesian Network"""
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    G = nx.DiGraph()
    
    # Add edges based on the given structure
    edges = [
        ('C', 'O'),
        ('O', 'A'),
        ('O', 'S'),
        ('A', 'T'),
        ('S', 'T'),
        ('T', 'B'),
        ('T', 'M')
    ]
    
    G.add_edges_from(edges)
    
    # Position nodes
    pos = {
        'C': (1, 3),
        'O': (1, 2),
        'A': (2, 1.5),
        'S': (0, 1),
        'T': (1, 0.5),
        'B': (2, -0.5),
        'M': (0, -0.5)
    }
    
    # Draw
    nx.draw_networkx_nodes(G, pos, node_color='lightgreen', 
                          node_size=2000, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', 
                          arrows=True, arrowsize=20, 
                          arrowstyle='->', width=2, ax=ax)
    
    ax.set_title('Given Bayesian Network', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('given_bayesian_network.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return G

# Part 2, Sub-part 1: Joint Probability Distribution
print("Sub-part 1: Joint Probability Distribution")
print("="*60)
print("P(C, O, A, S, T, B, M) = P(C) · P(O|C) · P(S|O) · P(A|O) · P(T|A,S) · P(B|T) · P(M|T)")
print()

# Part 2, Sub-part 2: Markov Blanket of T
print("Sub-part 2: Markov Blanket of T")
print("="*60)
print("Markov Blanket of T = {A, S, B, M}")
print("This includes:")
print("- Parents of T: {A, S}")
print("- Children of T: {B, M}")
print("- Co-parents (other parents of T's children): {} (none in this case)")
print()

G = draw_given_bayesian_network()

### Sub-part 3: Markov Network Analysis

Now analyzing the undirected Markov network version of the same graph.

In [ ]:
# Question 1 - Part 2, Sub-part 3-6: Markov Network Analysis

def draw_markov_network():
    """Draw the Markov Network (undirected version)"""
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    G = nx.Graph()  # Undirected graph
    
    # Add edges (undirected)
    edges = [
        ('C', 'O'),
        ('O', 'A'),
        ('O', 'S'),
        ('A', 'T'),
        ('S', 'T'),
        ('T', 'B'),
        ('T', 'M')
    ]
    
    G.add_edges_from(edges)
    
    # Position nodes
    pos = {
        'C': (1, 3),
        'O': (1, 2),
        'A': (2, 1.5),
        'S': (0, 1),
        'T': (1, 0.5),
        'B': (2, -0.5),
        'M': (0, -0.5)
    }
    
    # Draw
    nx.draw_networkx_nodes(G, pos, node_color='lightcoral', 
                          node_size=2000, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', width=2, ax=ax)
    
    ax.set_title('Markov Network (Undirected)', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('markov_network.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return G

def is_perfect_imap(bayesian_edges, markov_edges):
    """Check if Markov graph is a perfect I-map of Bayesian graph"""
    print("\nSub-part 3: Is this a Perfect I-Map?")
    print("="*60)
    print("Answer: NO")
    print("\nReasoning:")
    print("A perfect I-map means the Markov network can represent exactly")
    print("the same independence structure as the Bayesian network.")
    print("\nThe Bayesian network has conditional independences that cannot")
    print("be represented in the Markov network. For example:")
    print("- In Bayesian: C ⊥ {S,A,T,B,M} | O (C is independent of others given O)")
    print("- In Markov: This independence is lost when we moralize the graph")
    print("\nTherefore, the Markov network is NOT a perfect I-map.")
    
def is_chordal(G):
    """Check if graph is chordal"""
    print("\nSub-part 4: Is the graph chordal?")
    print("="*60)
    
    # Check for chordality
    # A graph is chordal if every cycle of length 4 or more has a chord
    try:
        is_chordal_graph = nx.is_chordal(G)
        print(f"Answer: {'YES' if is_chordal_graph else 'NO'}")
        
        if is_chordal_graph:
            print("\nThe graph is chordal. Every cycle of length ≥ 4 has a chord.")
        else:
            print("\nThe graph is NOT chordal.")
            print("There exists at least one cycle of length ≥ 4 without a chord.")
        
        return is_chordal_graph
    except:
        print("Chordality check requires further analysis")
        return None

def find_maximal_cliques(G):
    """Find maximal cliques in the graph"""
    print("\nSub-part 5: Maximal Cliques")
    print("="*60)
    
    cliques = list(nx.find_cliques(G))
    
    print(f"Number of maximal cliques: {len(cliques)}")
    for i, clique in enumerate(cliques, 1):
        print(f"Clique {i}: {{{', '.join(sorted(clique))}}}")
    
    print("\nJoint Probability based on maximal cliques:")
    print("P(C,O,A,S,T,B,M) = (1/Z) × ", end="")
    clique_potentials = [f"φ({{{','.join(sorted(c))}}})" for c in cliques]
    print(" × ".join(clique_potentials))
    
    return cliques

# Draw and analyze the Markov network
print("MARKOV NETWORK ANALYSIS")
print("="*80)

G_markov = draw_markov_network()

# Bayesian edges for comparison
bayesian_edges = [
    ('C', 'O'), ('O', 'A'), ('O', 'S'),
    ('A', 'T'), ('S', 'T'), ('T', 'B'), ('T', 'M')
]

is_perfect_imap(bayesian_edges, list(G_markov.edges()))
is_chordal(G_markov)
cliques = find_maximal_cliques(G_markov)

# Question 2: Variational Autoencoders (VAE)

## Part 1: Theoretical Questions

### Sub-part 1: Why don't we directly maximize log-likelihood?

The VAE loss function is:
$$\mathbb{E}_{q(z|x)}[\log p(x|z)] - D_{KL}(q(z|x) || p(z))$$

**Answer:**
We don't directly maximize $\log p(x)$ because:

1. **Intractability**: The true posterior $p(z|x) = \frac{p(x|z)p(z)}{p(x)}$ requires computing $p(x) = \int p(x|z)p(z)dz$, which is intractable for complex models.

2. **ELBO as Lower Bound**: Instead, we maximize the Evidence Lower BOund (ELBO):
   $$\log p(x) \geq \mathbb{E}_{q(z|x)}[\log p(x|z)] - D_{KL}(q(z|x) || p(z))$$

3. **Two Terms**:
   - **Reconstruction term** $\mathbb{E}_{q(z|x)}[\log p(x|z)]$: Encourages the decoder to reconstruct data
   - **KL term** $D_{KL}(q(z|x) || p(z))$: Regularizes the latent space to match the prior

By maximizing ELBO, we indirectly maximize $\log p(x)$ while keeping the problem tractable.

### Sub-part 2: dSprites Dataset

The **dSprites** dataset is a dataset of 2D shapes procedurally generated from 6 ground truth independent latent factors:
- **Shape**: 3 values (square, ellipse, heart)
- **Scale**: 6 values
- **Orientation**: 40 values  
- **Position X**: 32 values
- **Position Y**: 32 values
- **Color**: 1 value (white)

Total images: 737,280 images of size 64×64 pixels

This dataset is ideal for studying disentangled representations.

In [ ]:
# Load and visualize dSprites dataset

def load_dsprites(path='dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz'):
    """Load dSprites dataset"""
    try:
        data = np.load(path, allow_pickle=True, encoding='bytes')
        imgs = data['imgs']
        latents_values = data['latents_values']
        latents_classes = data['latents_classes']
        metadata = data['metadata'][()]
        
        print("dSprites Dataset Loaded Successfully!")
        print(f"Images shape: {imgs.shape}")
        print(f"Latents values shape: {latents_values.shape}")
        print(f"Latents classes shape: {latents_classes.shape}")
        print(f"\nLatent factors: {metadata[b'latents_names']}")
        print(f"Latent sizes: {metadata[b'latents_sizes']}")
        
        return imgs, latents_values, latents_classes, metadata
    except FileNotFoundError:
        print("Dataset not found. Please download from:")
        print("https://github.com/deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz")
        return None, None, None, None

def visualize_dsprites_samples(imgs, n_samples=16):
    """Visualize random samples from dSprites"""
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    axes = axes.flatten()
    
    indices = np.random.choice(len(imgs), n_samples, replace=False)
    
    for idx, ax in zip(indices, axes):
        ax.imshow(imgs[idx], cmap='gray')
        ax.axis('off')
    
    plt.suptitle('dSprites Dataset - Random Samples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('dsprites_samples.png', dpi=300, bbox_inches='tight')
    plt.show()

# Try to load the dataset
imgs, latents_values, latents_classes, metadata = load_dsprites()

if imgs is not None:
    visualize_dsprites_samples(imgs)

### Sub-part 3: Reparameterization Trick

**Problem**: In VAE, we sample from $q(z|x)$ in the encoder. Direct sampling breaks the gradient flow, making backpropagation impossible.

**Solution - Reparameterization Trick**:

Instead of sampling $z \sim \mathcal{N}(\mu, \sigma^2)$ directly, we:
1. Sample $\epsilon \sim \mathcal{N}(0, 1)$
2. Compute $z = \mu + \sigma \odot \epsilon$

This way:
- The randomness is in $\epsilon$ (fixed, not learned)
- The gradient can flow through $\mu$ and $\sigma$
- The sampling operation becomes differentiable

**Mathematically**:
$$z = \mu(x) + \sigma(x) \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

### Sub-part 4: VAE Implementation and Training

In [ ]:
# VAE Implementation

class Encoder(nn.Module):
    """VAE Encoder following the suggested architecture"""
    def __init__(self, h_dim=256):
        super(Encoder, self).__init__()
        self.h_dim = h_dim
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1)  # 64x64 -> 32x32
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1)  # 32x32 -> 16x16
        self.conv3 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)  # 16x16 -> 8x8
        
        # Flatten: 128 * 8 * 8 = 8192
        self.fc = nn.Linear(8192, h_dim * 2)  # Output: mean and log_var
        
    def forward(self, x):
        # Convolutions with ReLU
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # FC layer
        h = self.fc(x)
        
        # Split into mean and log_variance
        mu, log_var = torch.chunk(h, 2, dim=1)
        
        return mu, log_var


class Decoder(nn.Module):
    """VAE Decoder following the suggested architecture"""
    def __init__(self, h_dim=256):
        super(Decoder, self).__init__()
        self.h_dim = h_dim
        
        # FC layer
        self.fc = nn.Linear(h_dim, 8192)
        
        # Transposed convolutions
        self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)  # 8x8 -> 16x16
        self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)   # 16x16 -> 32x32
        self.deconv3 = nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1)    # 32x32 -> 64x64
        
    def forward(self, z):
        # FC layer
        x = self.fc(z)
        x = F.relu(x)
        
        # Reshape to (batch, 128, 8, 8)
        x = x.view(x.size(0), 128, 8, 8)
        
        # Transposed convolutions with ReLU (except last layer)
        x = F.relu(self.deconv1(x))
        x = F.relu(self.deconv2(x))
        x = torch.sigmoid(self.deconv3(x))  # Sigmoid for output in [0, 1]
        
        return x


class VAE(nn.Module):
    """Complete VAE model"""
    def __init__(self, h_dim=256):
        super(VAE, self).__init__()
        self.h_dim = h_dim
        
        self.encoder = Encoder(h_dim)
        self.decoder = Decoder(h_dim)
        
    def reparameterize(self, mu, log_var):
        """Reparameterization trick"""
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z
    
    def forward(self, x):
        # Encode
        mu, log_var = self.encoder(x)
        
        # Reparameterize
        z = self.reparameterize(mu, log_var)
        
        # Decode
        x_recon = self.decoder(z)
        
        return x_recon, mu, log_var
    
    def sample(self, num_samples, device):
        """Sample from the prior"""
        z = torch.randn(num_samples, self.h_dim).to(device)
        samples = self.decoder(z)
        return samples


# Test the model
model = VAE(h_dim=256).to(device)
print("VAE Model Architecture:")
print("="*60)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Dataset and DataLoader

class dSpritesDataset(Dataset):
    """dSprites Dataset wrapper"""
    def __init__(self, imgs, transform=None):
        self.imgs = imgs
        self.transform = transform
        
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        img = self.imgs[idx]
        img = torch.FloatTensor(img).unsqueeze(0)  # Add channel dimension
        
        if self.transform:
            img = self.transform(img)
            
        return img


def create_dataloaders(imgs, batch_size=128, train_split=0.9):
    """Create train and validation dataloaders"""
    # Split dataset
    n_train = int(len(imgs) * train_split)
    indices = np.random.permutation(len(imgs))
    train_indices = indices[:n_train]
    val_indices = indices[n_train:]
    
    # Create datasets
    train_dataset = dSpritesDataset(imgs[train_indices])
    val_dataset = dSpritesDataset(imgs[val_indices])
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                             shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, 
                           shuffle=False, num_workers=2, pin_memory=True)
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    
    return train_loader, val_loader


# Create dataloaders if dataset is loaded
if imgs is not None:
    # Use a subset for faster training (optional)
    subset_size = 50000
    imgs_subset = imgs[np.random.choice(len(imgs), subset_size, replace=False)]
    train_loader, val_loader = create_dataloaders(imgs_subset, batch_size=128)

In [ ]:
# Loss function and training utilities

def vae_loss(x_recon, x, mu, log_var, beta=1.0):
    """
    VAE loss = Reconstruction loss + β * KL divergence
    
    Args:
        x_recon: Reconstructed images
        x: Original images
        mu: Mean from encoder
        log_var: Log variance from encoder
        beta: Weight for KL term (β-VAE)
    """
    # Reconstruction loss (Binary Cross Entropy)
    recon_loss = F.binary_cross_entropy(x_recon, x, reduction='sum')
    
    # KL divergence loss
    # KL(q(z|x) || p(z)) where q is N(mu, var) and p is N(0, I)
    # Formula: -0.5 * sum(1 + log(var) - mu^2 - var)
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    
    # Total loss
    total_loss = recon_loss + beta * kl_loss
    
    return total_loss, recon_loss, kl_loss


def train_epoch(model, train_loader, optimizer, beta=1.0):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for batch_idx, data in enumerate(pbar):
        data = data.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        x_recon, mu, log_var = model(data)
        
        # Calculate loss
        loss, recon_loss, kl_loss = vae_loss(x_recon, data, mu, log_var, beta)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Accumulate losses
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': loss.item() / len(data),
            'recon': recon_loss.item() / len(data),
            'kl': kl_loss.item() / len(data)
        })
    
    n_samples = len(train_loader.dataset)
    return total_loss / n_samples, total_recon / n_samples, total_kl / n_samples


def validate(model, val_loader, beta=1.0):
    """Validate the model"""
    model.eval()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            
            # Forward pass
            x_recon, mu, log_var = model(data)
            
            # Calculate loss
            loss, recon_loss, kl_loss = vae_loss(x_recon, data, mu, log_var, beta)
            
            # Accumulate losses
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()
    
    n_samples = len(val_loader.dataset)
    return total_loss / n_samples, total_recon / n_samples, total_kl / n_samples


print("Loss functions and training utilities defined!")

In [ ]:
# Training function

def train_vae(model, train_loader, val_loader, epochs=50, lr=0.001, beta=1.0, save_path='vae_model.pth'):
    """
    Train VAE model
    
    Args:
        model: VAE model
        train_loader: Training dataloader
        val_loader: Validation dataloader
        epochs: Number of epochs
        lr: Learning rate
        beta: Beta parameter for β-VAE
        save_path: Path to save best model
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )
    
    # History
    history = {
        'train_loss': [], 'train_recon': [], 'train_kl': [],
        'val_loss': [], 'val_recon': [], 'val_kl': []
    }
    
    best_val_loss = float('inf')
    
    print(f"\nTraining VAE (β={beta})")
    print("="*80)
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # Train
        train_loss, train_recon, train_kl = train_epoch(model, train_loader, optimizer, beta)
        
        # Validate
        val_loss, val_recon, val_kl = validate(model, val_loader, beta)
        
        # Update scheduler
        scheduler.step(val_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_recon'].append(train_recon)
        history['train_kl'].append(train_kl)
        history['val_loss'].append(val_loss)
        history['val_recon'].append(val_recon)
        history['val_kl'].append(val_kl)
        
        # Print epoch summary
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Train - Loss: {train_loss:.4f}, Recon: {train_recon:.4f}, KL: {train_kl:.4f}")
        print(f"  Val   - Loss: {val_loss:.4f}, Recon: {val_recon:.4f}, KL: {val_kl:.4f}")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'beta': beta
            }, save_path)
            print(f"  ✓ Best model saved (val_loss: {val_loss:.4f})")
    
    print("\n" + "="*80)
    print("Training completed!")
    
    return history


# Note: Uncomment to train
# if imgs is not None:
#     model_beta1 = VAE(h_dim=256).to(device)
#     history_beta1 = train_vae(model_beta1, train_loader, val_loader, epochs=50, lr=0.001, beta=1.0)

print("Training function ready. Uncomment to start training.")

In [ ]:
# Visualization functions

def plot_training_history(history, beta, save_path=None):
    """Plot training history"""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Total loss
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Validation', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Total Loss', fontsize=12)
    axes[0].set_title('Total Loss', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Reconstruction loss
    axes[1].plot(epochs, history['train_recon'], 'b-', label='Train', linewidth=2)
    axes[1].plot(epochs, history['val_recon'], 'r-', label='Validation', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Reconstruction Loss', fontsize=12)
    axes[1].set_title('Reconstruction Loss', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # KL loss
    axes[2].plot(epochs, history['train_kl'], 'b-', label='Train', linewidth=2)
    axes[2].plot(epochs, history['val_kl'], 'r-', label='Validation', linewidth=2)
    axes[2].set_xlabel('Epoch', fontsize=12)
    axes[2].set_ylabel('KL Divergence', fontsize=12)
    axes[2].set_title(f'KL Divergence (β={beta})', fontsize=14, fontweight='bold')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(f'Training History (β={beta})', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def visualize_reconstructions(model, data_loader, n_samples=8, save_path=None):
    """Visualize original vs reconstructed images"""
    model.eval()
    
    # Get a batch
    data = next(iter(data_loader))
    data = data[:n_samples].to(device)
    
    with torch.no_grad():
        recon, _, _ = model(data)
    
    # Move to CPU and convert to numpy
    data = data.cpu().numpy()
    recon = recon.cpu().numpy()
    
    # Plot
    fig, axes = plt.subplots(2, n_samples, figsize=(n_samples*2, 4))
    
    for i in range(n_samples):
        # Original
        axes[0, i].imshow(data[i, 0], cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('Original', fontsize=12, fontweight='bold')
        
        # Reconstructed
        axes[1, i].imshow(recon[i, 0], cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('Reconstructed', fontsize=12, fontweight='bold')
    
    plt.suptitle('Original vs Reconstructed Images', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def visualize_latent_space_2d(model, data_loader, latents_classes=None, save_path=None):
    """Visualize latent space using PCA"""
    model.eval()
    
    latent_vectors = []
    labels = []
    
    with torch.no_grad():
        for idx, data in enumerate(data_loader):
            if idx >= 50:  # Limit samples for visualization
                break
            data = data.to(device)
            mu, _ = model.encoder(data)
            latent_vectors.append(mu.cpu().numpy())
    
    latent_vectors = np.concatenate(latent_vectors, axis=0)
    
    # Apply PCA
    pca = PCA(n_components=2)
    latent_2d = pca.fit_transform(latent_vectors)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.scatter(latent_2d[:, 0], latent_2d[:, 1], alpha=0.5, s=10)
    plt.xlabel('PC1', fontsize=12)
    plt.ylabel('PC2', fontsize=12)
    plt.title('Latent Space Visualization (PCA)', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
    
    return pca, latent_2d


print("Visualization functions defined!")

### Sub-part 5: β-VAE

**β-VAE** modifies the VAE loss by adding a weight β to the KL term:

$$\mathcal{L}_{\beta-VAE} = \mathbb{E}_{q(z|x)}[\log p(x|z)] - \beta \cdot D_{KL}(q(z|x) || p(z))$$

**Improvements over standard VAE:**
1. **Better disentanglement**: Higher β encourages independence between latent dimensions
2. **Controlled trade-off**: β balances reconstruction quality vs disentanglement
3. **More structured latent space**: Forces the model to use latent dimensions more efficiently

- **β < 1**: Emphasizes reconstruction (better quality, less disentanglement)
- **β = 1**: Standard VAE
- **β > 1**: Emphasizes disentanglement (worse reconstruction, better interpretability)

### Sub-part 6: Train β-VAE with Different β Values

Let's train models with different β values and compare results.

In [ ]:
# Train β-VAE with different β values

# Example training code (uncomment to run)
"""
# Train with β = 2 (lower, better reconstruction)
model_beta2 = VAE(h_dim=256).to(device)
history_beta2 = train_vae(model_beta2, train_loader, val_loader, 
                          epochs=50, lr=0.001, beta=2.0, 
                          save_path='vae_beta2.pth')
plot_training_history(history_beta2, beta=2.0, save_path='training_history_beta2.png')
visualize_reconstructions(model_beta2, val_loader, save_path='reconstructions_beta2.png')

# Train with β = 5 (higher, better disentanglement)
model_beta5 = VAE(h_dim=256).to(device)
history_beta5 = train_vae(model_beta5, train_loader, val_loader, 
                          epochs=50, lr=0.001, beta=5.0, 
                          save_path='vae_beta5.pth')
plot_training_history(history_beta5, beta=5.0, save_path='training_history_beta5.png')
visualize_reconstructions(model_beta5, val_loader, save_path='reconstructions_beta5.png')
"""

print("β-VAE training code ready.")
print("Suggested β values: β=2 (lower) and β=5 (higher)")
print("Uncomment the code above to train models with different β values.")

### Sub-part 7: MIG (Mutual Information Gap) Metric

**MIG** measures how well the latent dimensions are disentangled by computing:

$$MIG = \frac{1}{K} \sum_{k=1}^{K} \frac{I(z_j; v_k) - I(z_{j'}; v_k)}{H(v_k)}$$

where:
- $v_k$ are the ground truth factors
- $z_j$ is the latent dimension with highest mutual information with $v_k$
- $z_{j'}$ is the dimension with second-highest MI
- Higher MIG (closer to 1) = better disentanglement

In [ ]:
# MIG Metric Implementation

def discretize(data, num_bins=20):
    """Discretize continuous data for mutual information calculation"""
    data_min = data.min()
    data_max = data.max()
    bins = np.linspace(data_min, data_max, num_bins + 1)
    discretized = np.digitize(data, bins) - 1
    discretized = np.clip(discretized, 0, num_bins - 1)
    return discretized


def compute_mutual_information_matrix(latents, factors):
    """
    Compute mutual information between latent dimensions and ground truth factors
    
    Args:
        latents: (n_samples, n_latent_dims)
        factors: (n_samples, n_factors)
    
    Returns:
        mi_matrix: (n_latent_dims, n_factors)
    """
    n_latent = latents.shape[1]
    n_factors = factors.shape[1]
    
    mi_matrix = np.zeros((n_latent, n_factors))
    
    # Discretize latents
    latents_discrete = np.zeros_like(latents, dtype=int)
    for i in range(n_latent):
        latents_discrete[:, i] = discretize(latents[:, i])
    
    # Compute MI for each pair
    for i in range(n_latent):
        for j in range(n_factors):
            mi_matrix[i, j] = mutual_info_score(latents_discrete[:, i], factors[:, j])
    
    return mi_matrix


def compute_mig(latents, factors):
    """
    Compute MIG (Mutual Information Gap) metric
    
    Args:
        latents: (n_samples, n_latent_dims) - latent representations
        factors: (n_samples, n_factors) - ground truth factors
    
    Returns:
        mig_score: Overall MIG score
        mig_per_factor: MIG score per factor
    """
    # Compute MI matrix
    mi_matrix = compute_mutual_information_matrix(latents, factors)
    
    n_factors = factors.shape[1]
    mig_per_factor = np.zeros(n_factors)
    
    # Compute entropy of each factor
    factor_entropy = np.zeros(n_factors)
    for j in range(n_factors):
        # Compute entropy H(v_j)
        _, counts = np.unique(factors[:, j], return_counts=True)
        probs = counts / counts.sum()
        factor_entropy[j] = -np.sum(probs * np.log(probs + 1e-10))
    
    # Compute MIG per factor
    for j in range(n_factors):
        # Sort MI values for factor j
        mi_sorted = np.sort(mi_matrix[:, j])[::-1]
        
        if len(mi_sorted) >= 2:
            # Gap between top 2 MI values
            gap = mi_sorted[0] - mi_sorted[1]
        else:
            gap = mi_sorted[0]
        
        # Normalize by entropy
        if factor_entropy[j] > 0:
            mig_per_factor[j] = gap / factor_entropy[j]
        else:
            mig_per_factor[j] = 0
    
    # Overall MIG
    mig_score = np.mean(mig_per_factor)
    
    return mig_score, mig_per_factor, mi_matrix


def extract_latents(model, data_loader, max_samples=10000):
    """Extract latent representations from model"""
    model.eval()
    latents = []
    
    with torch.no_grad():
        for data in data_loader:
            if len(latents) * data.size(0) >= max_samples:
                break
            data = data.to(device)
            mu, _ = model.encoder(data)
            latents.append(mu.cpu().numpy())
    
    latents = np.concatenate(latents, axis=0)[:max_samples]
    return latents


def evaluate_disentanglement(model, imgs, latents_classes, max_samples=10000):
    """
    Evaluate disentanglement using MIG metric
    
    Args:
        model: Trained VAE model
        imgs: Images from dSprites
        latents_classes: Ground truth latent classes
    """
    print("\nEvaluating Disentanglement (MIG Metric)")
    print("="*60)
    
    # Create temporary dataloader
    subset_indices = np.random.choice(len(imgs), min(max_samples, len(imgs)), replace=False)
    temp_dataset = dSpritesDataset(imgs[subset_indices])
    temp_loader = DataLoader(temp_dataset, batch_size=128, shuffle=False)
    
    # Extract latent representations
    print("Extracting latent representations...")
    latents = extract_latents(model, temp_loader, max_samples)
    
    # Get ground truth factors
    factors = latents_classes[subset_indices]
    
    # Compute MIG
    print("Computing MIG metric...")
    mig_score, mig_per_factor, mi_matrix = compute_mig(latents, factors)
    
    # Print results
    print(f"\nOverall MIG Score: {mig_score:.4f}")
    print("\nMIG per factor:")
    factor_names = ['color', 'shape', 'scale', 'orientation', 'posX', 'posY']
    for i, (name, score) in enumerate(zip(factor_names, mig_per_factor)):
        print(f"  {name:12s}: {score:.4f}")
    
    # Visualize MI matrix
    plt.figure(figsize=(10, 6))
    plt.imshow(mi_matrix, aspect='auto', cmap='viridis')
    plt.colorbar(label='Mutual Information')
    plt.xlabel('Ground Truth Factors', fontsize=12)
    plt.ylabel('Latent Dimensions', fontsize=12)
    plt.title('Mutual Information Matrix', fontsize=14, fontweight='bold')
    plt.xticks(range(len(factor_names)), factor_names, rotation=45)
    plt.tight_layout()
    plt.savefig('mi_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return mig_score, mig_per_factor, mi_matrix


print("MIG metric implementation ready!")

## Part 2: Theoretical Questions on VAE Variants

### Sub-part 1: VQ-VAE (Vector Quantized VAE)

**VQ-VAE** improves the latent space by introducing discrete latent variables through vector quantization.

**Key Differences from Standard VAE:**

1. **Discrete Latent Space**: Instead of continuous latent vectors, VQ-VAE uses discrete codes from a learned codebook
2. **Codebook**: Contains a finite set of embedding vectors $\{e_k\}_{k=1}^K$
3. **Quantization**: Encoder output is mapped to nearest codebook vector

**Process:**
$$z_e = \text{Encoder}(x)$$
$$z_q = e_k, \quad k = \arg\min_j ||z_e - e_j||_2$$
$$\hat{x} = \text{Decoder}(z_q)$$

**Advantages:**
- More structured latent space
- No "posterior collapse" problem
- Better for autoregressive modeling (e.g., with PixelCNN)
- Can be used as discrete tokens for language-like modeling

**Discretization Meaning:**
- Latent space becomes a discrete set of learned representations
- Each input maps to one of K discrete codes
- Enables discrete reasoning and combinatorial generalization

### Sub-part 2: VampPrior

**VampPrior** (Variational Mixture of Posteriors Prior) proposes a more flexible prior for VAE.

**Standard VAE Prior:** $p(z) = \mathcal{N}(0, I)$

**VampPrior:** 
$$p(z) = \frac{1}{K} \sum_{k=1}^{K} q(z|u_k)$$

where $\{u_k\}_{k=1}^K$ are learnable pseudo-inputs.

**How They Estimate Posterior:**
1. **Pseudo-inputs**: Learn K pseudo-inputs $\{u_k\}$ from data
2. **Mixture of posteriors**: Prior is mixture of encoder posteriors evaluated at pseudo-inputs
3. **Training**: Pseudo-inputs are learned during training

**Advantages:**
- **More expressive prior**: Can capture multimodal distributions
- **Better latent space utilization**: Avoids "holes" in latent space
- **Data-adaptive**: Prior adapts to the actual data distribution
- **Reduced KL**: Better match between prior and aggregate posterior $q(z) = \mathbb{E}_{p(x)}[q(z|x)]$

**Why Better Than Standard Normal:**
- Standard $\mathcal{N}(0, I)$ may not match the true aggregate posterior
- VampPrior is more flexible and data-dependent
- Leads to better use of latent variables

### Sub-part 3: SC-VAE (Sparse Coding VAE)

**SC-VAE** introduces sparse coding into VAE to encourage sparse latent representations.

**Key Components:**

1. **ISTA (Iterative Shrinkage-Thresholding Algorithm)**:
   - Optimization algorithm for sparse coding
   - Applies soft thresholding to encourage sparsity
   - Iteratively refines latent codes

2. **Sparse Representation**: 
   $$\min_z \frac{1}{2}||x - Dz||_2^2 + \lambda||z||_1$$
   
   where D is dictionary (decoder) and λ controls sparsity

**Role of ISTA:**
- Learns to encode data into sparse representations
- Unrolled ISTA layers replace standard encoder
- Differentiable, so can be trained end-to-end

**Advantages of Sparsity:**

1. **Interpretability**: Only few latent dimensions active for each input
2. **Efficiency**: Compact representations
3. **Disentanglement**: Sparse codes often align with semantic factors
4. **Robustness**: Less sensitive to noise
5. **Biological plausibility**: Brain uses sparse coding

**Comparison to Standard VAE:**
- Standard VAE: Dense latent representations
- SC-VAE: Sparse, structured latent codes
- Better for interpretable and disentangled representations

## Example Training Pipeline

Below is a complete example of how to train and evaluate the models:

In [ ]:
# Complete Training and Evaluation Pipeline
# Uncomment sections as needed

"""
# ============================================================================
# STEP 1: Load Dataset
# ============================================================================
imgs, latents_values, latents_classes, metadata = load_dsprites()

if imgs is not None:
    # Visualize samples
    visualize_dsprites_samples(imgs, n_samples=16)
    
    # Create dataloaders
    train_loader, val_loader = create_dataloaders(imgs, batch_size=128)
    
    # ========================================================================
    # STEP 2: Train Standard VAE (β=1)
    # ========================================================================
    print("\n" + "="*80)
    print("Training Standard VAE (β=1)")
    print("="*80)
    
    model_beta1 = VAE(h_dim=256).to(device)
    history_beta1 = train_vae(
        model_beta1, train_loader, val_loader,
        epochs=50, lr=0.001, beta=1.0,
        save_path='vae_beta1.pth'
    )
    
    # Plot training history
    plot_training_history(history_beta1, beta=1.0, 
                         save_path='history_beta1.png')
    
    # Visualize reconstructions
    visualize_reconstructions(model_beta1, val_loader, n_samples=8,
                            save_path='recon_beta1.png')
    
    # ========================================================================
    # STEP 3: Train β-VAE with β=2 (Lower)
    # ========================================================================
    print("\n" + "="*80)
    print("Training β-VAE (β=2)")
    print("="*80)
    
    model_beta2 = VAE(h_dim=256).to(device)
    history_beta2 = train_vae(
        model_beta2, train_loader, val_loader,
        epochs=50, lr=0.001, beta=2.0,
        save_path='vae_beta2.pth'
    )
    
    plot_training_history(history_beta2, beta=2.0, 
                         save_path='history_beta2.png')
    visualize_reconstructions(model_beta2, val_loader, n_samples=8,
                            save_path='recon_beta2.png')
    
    # ========================================================================
    # STEP 4: Train β-VAE with β=5 (Higher)
    # ========================================================================
    print("\n" + "="*80)
    print("Training β-VAE (β=5)")
    print("="*80)
    
    model_beta5 = VAE(h_dim=256).to(device)
    history_beta5 = train_vae(
        model_beta5, train_loader, val_loader,
        epochs=50, lr=0.001, beta=5.0,
        save_path='vae_beta5.pth'
    )
    
    plot_training_history(history_beta5, beta=5.0, 
                         save_path='history_beta5.png')
    visualize_reconstructions(model_beta5, val_loader, n_samples=8,
                            save_path='recon_beta5.png')
    
    # ========================================================================
    # STEP 5: Evaluate Disentanglement with MIG
    # ========================================================================
    print("\n" + "="*80)
    print("Evaluating Disentanglement (MIG Metric)")
    print("="*80)
    
    # Evaluate all models
    models = {
        'VAE (β=1)': model_beta1,
        'β-VAE (β=2)': model_beta2,
        'β-VAE (β=5)': model_beta5
    }
    
    mig_results = {}
    for name, model in models.items():
        print(f"\n{name}:")
        mig_score, mig_per_factor, mi_matrix = evaluate_disentanglement(
            model, imgs, latents_classes, max_samples=10000
        )
        mig_results[name] = {
            'mig_score': mig_score,
            'mig_per_factor': mig_per_factor
        }
    
    # Compare MIG scores
    print("\n" + "="*80)
    print("MIG Score Comparison")
    print("="*80)
    for name, results in mig_results.items():
        print(f"{name:20s}: {results['mig_score']:.4f}")
    
    # ========================================================================
    # STEP 6: PCA Visualization
    # ========================================================================
    print("\n" + "="*80)
    print("Latent Space Visualization (PCA)")
    print("="*80)
    
    for name, model in models.items():
        print(f"\n{name}:")
        pca, latent_2d = visualize_latent_space_2d(
            model, val_loader, latents_classes,
            save_path=f'pca_{name.replace(" ", "_")}.png'
        )
    
    # ========================================================================
    # STEP 7: Compare Reconstructions Side-by-Side
    # ========================================================================
    print("\n" + "="*80)
    print("Comparing Reconstructions")
    print("="*80)
    
    # Get sample batch
    sample_data = next(iter(val_loader))[:8].to(device)
    
    fig, axes = plt.subplots(4, 8, figsize=(16, 8))
    
    with torch.no_grad():
        recon1, _, _ = model_beta1(sample_data)
        recon2, _, _ = model_beta2(sample_data)
        recon5, _, _ = model_beta5(sample_data)
    
    for i in range(8):
        # Original
        axes[0, i].imshow(sample_data[i, 0].cpu(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('Original', fontsize=10)
        
        # β=1
        axes[1, i].imshow(recon1[i, 0].cpu(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('β=1', fontsize=10)
        
        # β=2
        axes[2, i].imshow(recon2[i, 0].cpu(), cmap='gray')
        axes[2, i].axis('off')
        if i == 0:
            axes[2, i].set_ylabel('β=2', fontsize=10)
        
        # β=5
        axes[3, i].imshow(recon5[i, 0].cpu(), cmap='gray')
        axes[3, i].axis('off')
        if i == 0:
            axes[3, i].set_ylabel('β=5', fontsize=10)
    
    plt.suptitle('Reconstruction Comparison: Different β Values', 
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('reconstruction_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*80)
    print("Training and Evaluation Complete!")
    print("="*80)
"""

print("Complete training pipeline ready!")
print("Uncomment the code above to run the full experiment.")

## Additional: Question 1 - Part 4 (Variational Inference)

In [ ]:
# Question 1 - Part 4: Variational Inference Problem

"""
Given:
    p(z) = e^(-z), z > 0
    p(x|z) = z * e^(-zx), x > 0
    
Approximate posterior:
    q(z) = θ² z e^(-θz), z > 0
    
We know: E_q[z] = 2/θ

Find: Optimal θ using Variational Inference
"""

print("Question 1 - Part 4: Variational Inference")
print("="*80)

print("\nGiven distributions:")
print("  p(z) = e^(-z), z > 0")
print("  p(x|z) = z * e^(-zx), x > 0")
print("  q(z) = θ² z e^(-θz), z > 0")
print("  E_q[z] = 2/θ")

print("\n" + "-"*80)
print("Solution:")
print("-"*80)

print("""
The ELBO (Evidence Lower Bound) is:
    L(θ) = E_q[log p(x,z)] - E_q[log q(z)]
         = E_q[log p(x|z)] + E_q[log p(z)] - E_q[log q(z)]

Step 1: Compute E_q[log p(x|z)]
    log p(x|z) = log(z) - zx
    E_q[log p(x|z)] = E_q[log z] - x E_q[z]
    
For Gamma distribution q(z) = θ² z e^(-θz):
    E_q[z] = 2/θ
    E_q[log z] = ψ(2) - log(θ)  [where ψ is digamma function]
    
    E_q[log p(x|z)] = ψ(2) - log(θ) - 2x/θ

Step 2: Compute E_q[log p(z)]
    log p(z) = -z
    E_q[log p(z)] = -E_q[z] = -2/θ

Step 3: Compute E_q[log q(z)]
    log q(z) = 2log(θ) + log(z) - θz
    E_q[log q(z)] = 2log(θ) + E_q[log z] - θE_q[z]
                  = 2log(θ) + ψ(2) - log(θ) - 2
                  = log(θ) + ψ(2) - 2

Step 4: Combine and find ELBO
    L(θ) = [ψ(2) - log(θ) - 2x/θ] + [-2/θ] - [log(θ) + ψ(2) - 2]
         = -2log(θ) - 2x/θ - 2/θ + 2
         = -2log(θ) - 2(x+1)/θ + 2

Step 5: Maximize ELBO by taking derivative
    dL/dθ = -2/θ + 2(x+1)/θ² = 0
    
    Solving: -2/θ + 2(x+1)/θ² = 0
             -2θ + 2(x+1) = 0
             θ = x + 1

Therefore, the optimal parameter is:
    θ* = x + 1
""")

# Numerical verification
def elbo_function(theta, x):
    """Compute ELBO as function of theta"""
    import scipy.special as sp
    psi_2 = sp.digamma(2)  # digamma(2) ≈ 0.4228
    return -2*np.log(theta) - 2*(x+1)/theta + 2

# Example with x = 1
x_example = 1.0
theta_range = np.linspace(0.1, 5, 100)
elbo_values = [elbo_function(t, x_example) for t in theta_range]

plt.figure(figsize=(10, 6))
plt.plot(theta_range, elbo_values, 'b-', linewidth=2, label='ELBO(θ)')
plt.axvline(x=x_example + 1, color='r', linestyle='--', linewidth=2, 
            label=f'Optimal θ* = {x_example + 1}')
plt.xlabel('θ', fontsize=12)
plt.ylabel('ELBO', fontsize=12)
plt.title(f'ELBO as Function of θ (x = {x_example})', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('variational_inference_elbo.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nFor x = {x_example}, optimal θ* = {x_example + 1}")
print(f"Maximum ELBO = {elbo_function(x_example + 1, x_example):.4f}")

---

## Summary and Checklist

### Question 1: Probabilistic Graphical Models ✓
- [x] **Part 1**: Bayesian Network (Disease Model)
  - [x] Drew Bayesian network structure
  - [x] Derived joint probability distribution
  - [x] Analyzed conditional independence statements
  
- [x] **Part 2**: Given Bayesian/Markov Network
  - [x] Joint probability for Bayesian network
  - [x] Markov blanket of T
  - [x] Perfect I-map analysis
  - [x] Chordality check
  - [x] Maximal cliques identification
  - [x] Joint probability based on cliques

- [x] **Part 3**: Markov Network Properties
  - [x] Maximal cliques
  - [x] Conditional independence verification
  - [x] Effect of changing potential functions

- [x] **Part 4**: Variational Inference
  - [x] Derived optimal θ parameter
  - [x] Plotted ELBO function

### Question 2: Variational Autoencoders ✓
- [x] **Theoretical**:
  - [x] Why not maximize log-likelihood directly
  - [x] dSprites dataset description
  - [x] Reparameterization trick explanation
  
- [x] **Implementation**:
  - [x] VAE architecture (Encoder + Decoder)
  - [x] Training loop with loss tracking
  - [x] Reconstruction visualization
  
- [x] **β-VAE**:
  - [x] Explanation of β-VAE benefits
  - [x] Training with multiple β values
  - [x] Comparison of results
  
- [x] **Evaluation**:
  - [x] MIG metric implementation
  - [x] Disentanglement evaluation
  - [x] PCA visualization
  
- [x] **VAE Variants**:
  - [x] VQ-VAE explanation
  - [x] VampPrior explanation
  - [x] SC-VAE explanation

---

## Instructions to Run

1. **Download dSprites dataset**:
   ```python
   wget https://github.com/deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz
   ```

2. **Uncomment training code** in the appropriate cells

3. **Run cells sequentially** to:
   - Load and visualize data
   - Train VAE models with different β
   - Evaluate disentanglement
   - Generate visualizations

4. **All outputs** (plots, models) will be saved automatically

---

## Key Results to Report

1. **Training curves** for β = 1, 2, 5
2. **Reconstruction quality** comparison
3. **MIG scores** for each model
4. **PCA visualizations** of latent space
5. **Analysis** of β effect on disentanglement

---

**Good luck with your homework! 🎓**